In [1]:
import cvxpy as cp
import numpy as np

In [2]:
# problem 

G0 = cp.Variable((2, 2), hermitian=True)
G1 = np.eye(2) - G0

In [3]:
N0 = cp.Variable((2, 2), hermitian=True)
N1 = np.eye(2) - N0

In [5]:
s = cp.Variable(nonneg=True)  # scalar, enforces s >= 0 directly

M0 = np.array([[0.8, 0],
               [0,   0.2]], dtype=complex)
M1 = np.eye(2) - M0

In [7]:
constraints = [
    G0 >> 0,
    G1 >> 0,
    N0 >> 0,
    N1 >> 0,
    M0 - s * N0 == (1 + s) * G0,
    M1 - s * N1 == (1 + s) * G1,
    s >= 0
] 

In [10]:
objective = cp.Minimize(s)
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.SCS)

DCPError: Problem does not follow DCP rules. Specifically:
The following constraints are not DCP:
[[0.8+0.j 0. +0.j]
 [0. +0.j 0.2+0.j]] + -(Promote(var7, (2, 2)) * var4) == Promote(1.0 + var7, (2, 2)) * var1 , because the following subexpressions are not:
|--  Promote(var7, (2, 2)) * var4
|--  Promote(1.0 + var7, (2, 2)) * var1
[[0.2+0.j 0. +0.j]
 [0. +0.j 0.8+0.j]] + -(Promote(var7, (2, 2)) * ([[1.00 0.00]
 [0.00 1.00]] + -var4)) == Promote(1.0 + var7, (2, 2)) * ([[1.00 0.00]
 [0.00 1.00]] + -var1) , because the following subexpressions are not:
|--  Promote(var7, (2, 2)) * ([[1.00 0.00]
 [0.00 1.00]] + -var4)
|--  Promote(1.0 + var7, (2, 2)) * ([[1.00 0.00]
 [0.00 1.00]] + -var1)

In [11]:
M0 = np.array([[0.8, 0], [0, 0.2]], dtype=complex)
M1 = np.eye(2) - M0

s  = cp.Variable(nonneg=True)
N0 = cp.Variable((2, 2), hermitian=True)  # represents s*N_0
G0 = cp.Variable((2, 2), hermitian=True)  # represents (1+s)*G_0

# derived quantities — all linear in variables
N1 = s * np.eye(2) - N0
G1 = (1 + s) * np.eye(2) - G0

constraints = [
    # marginalization (now linear)
    M0 + N0 == G0,
    M1 + N1 == G1,
    # PSD constraints on original N_a, G_a
    # N0_orig = N0/s >= 0 iff N0 >= 0 (s >= 0)
    N0 >> 0,
    N1 >> 0,
    G0 >> 0,
    G1 >> 0,
]

objective = cp.Minimize(s)
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.SCS)

print(f"Generalized robustness s*: {s.value:.6f}")


Generalized robustness s*: 0.000000


In [13]:
Mz = [np.array([[1, 0], [0, 0]]), np.array([[0, 0], [0, 1]])]
Mx = [0.5 * np.array([[1, 1], [1, 1]]), 0.5 * np.array([[1, -1], [-1, 1]])]
Nz = [cp.Variable((2, 2), hermitian=True), cp.Variable((2, 2), hermitian=True)]

Nx = [cp.Variable((2, 2), hermitian=True), cp.Variable((2, 2), hermitian=True)]

In [14]:
G = [[cp.Variable((2, 2), hermitian=True), cp.Variable((2, 2), hermitian=True)], [cp.Variable((2, 2), hermitian=True), cp.Variable((2, 2), hermitian=True)]]

for a in range(2):
    constraints.append(Mz[a] + Nz[a] == G[a][0] + G[a][1])
    constraints.append(Nz[a] >> 0)
    constraints.append(G[a][0] >> 0)
    constraints.append(G[a][1] >> 0)

for b in range(2):
    constraints.append(Mx[b] + Nx[b] == G[0][b] + G[1][b])
    constraints.append(Nx[b] >> 0)

constraints.append(s >= 0)
constraints.append(G[0][0] + G[1][0] + G[1][1] + G[0][1] == np.eye(2) * (1 + s))

constraints.append(Nz[0] + Nz[1] == s * np.eye(2))
constraints.append(Nx[0] + Nx[1] == s * np.eye(2))

objective = cp.Minimize(s)

In [15]:
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.SCS)

np.float64(0.17157287129397783)

In [16]:
for a in range(2):
    for b in range(2):
        print(f"G[{a}][{b}] eigenvalues: {np.linalg.eigvalsh(G[a][b].value).round(4)}")
print(f"Generalized robustness s*: {s.value:.6f}")

G[0][0] eigenvalues: [-0.      0.5858]
G[0][1] eigenvalues: [-0.      0.5858]
G[1][0] eigenvalues: [-0.      0.5858]
G[1][1] eigenvalues: [-0.      0.5858]
Generalized robustness s*: 0.171573
